# M1 · Supervised learning

_AFP-AI · Domain 0 · ML Foundations_

**Learn a function from labeled examples, then judge it on data it has never seen.**

We build a binary click-style classifier end to end: split the data honestly, fit a model, and read the **generalization gap** between train and validation. Run each cell top to bottom. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

rng = np.random.default_rng(0)

## First, look at the data

Each row is one impression: a few numeric features and a binary label `clicked`. We make the positive rate low (about 6%) so it behaves like a real ads dataset.

In [ ]:
# Synthetic impressions: features drive a true click probability, then we sample labels.
n = 4000
X = rng.normal(size=(n, 3))

# True log-odds is linear in the features (with an offset that makes clicks rare).
true_w = np.array([1.2, -0.8, 0.5])
logits = X @ true_w - 3.0
p_true = 1.0 / (1.0 + np.exp(-logits))

y = (rng.random(n) < p_true).astype(int)

print("rows:", n, " positive rate:", round(y.mean(), 4))

## The model, in one formula

Logistic regression predicts a probability by squashing a linear score through the sigmoid:

$$p = \sigma(w^\top x + b) = \frac{1}{1 + e^{-(w^\top x + b)}}$$

and it is fit by minimizing **log loss** $\ell = -[y\log p + (1-y)\log(1-p)]$.

### Step 1 - Split the data honestly

We hold out a validation set the fitting never sees. `stratify=y` keeps the same rare-positive rate in both folds, so a split does not accidentally starve one side of positives.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

print("train positives:", y_train.mean().round(4))
print("val   positives:", y_val.mean().round(4))

# Stratification keeps the two rates close.
assert abs(y_train.mean() - y_val.mean()) < 0.02

### Step 2 - Fit and score on both splits

We fit on train only, then measure AUC and log loss on each split. The number that matters is the **gap** between train and validation.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

p_train = model.predict_proba(X_train)[:, 1]
p_val = model.predict_proba(X_val)[:, 1]

auc_train = roc_auc_score(y_train, p_train)
auc_val = roc_auc_score(y_val, p_val)

print("train AUC:", round(auc_train, 3))
print("val   AUC:", round(auc_val, 3))
print("gap     :", round(auc_train - auc_val, 3))

### Step 3 - A well-specified linear model barely overfits

Because the true relationship is linear, train and validation AUC should land close together - a small gap. We assert the gap is modest as a sanity check.

In [ ]:
gap = auc_train - auc_val

# A correctly-specified model on enough data generalizes: the gap stays small.
assert gap < 0.05

print("generalization gap is small:", round(gap, 3))

## Visualize the generalization gap

The bars compare train vs validation AUC. When you deliberately overfit (few rows, huge capacity), the two bars pull apart - that spreading gap is the thing supervised learning is always fighting.

In [ ]:
labels = ["train", "val"]
values = [auc_train, auc_val]

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(labels, values, color=["#4c78a8", "#f58518"])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("AUC")
ax.set_title("train vs validation AUC")
plt.show()

## Practice

Try each in the empty cell below it.

1. Shrink the training set to 60 rows and add 20 noise features. Re-fit and watch the gap grow - reproduce overfitting.
2. Add class-weighting (`LogisticRegression(class_weight="balanced")`) and compare val AUC.
3. Replace AUC with `log_loss` and compare the train/val gap under that metric instead.

In [ ]:
# Your turn:
